# Text Representation (BoW & TF-IDF)

> 📘 **Python Mastery** · Module 16 — NLP · Lesson 3/5

Clean tokens still mean nothing to a model — this lesson turns words into numbers with Bag-of-Words and TF-IDF, builds a tiny search engine out of cosine similarity, and finishes with a working SMS spam classifier.

## 🎯 Learning Objectives

- Explain why text must become numeric vectors and compute how sparse one-hot encoding really is
- Create document-term matrices with `CountVectorizer` and inspect `.vocabulary_` and `.toarray()`
- Compute TF and IDF **by hand**, then verify your arithmetic against `TfidfVectorizer`
- Implement cosine similarity from the formula and rank documents for a query — a mini search engine
- Choose analyzers wisely: word vs character n-grams for typos, `ngram_range=(1,2)` for word order
- Train and evaluate an SMS spam classifier inside a scikit-learn `Pipeline`, then inspect its errors

## 1. Computers Need Numbers

A model multiplies and adds — it cannot multiply `"battery"` by 3. So every word becomes a coordinate. The simplest scheme is **one-hot**: one slot per vocabulary word, `1` at the position of this word, `0` elsewhere. Immediately we feel the cost: real vocabularies hold tens of thousands of words, so vectors are almost entirely zeros.

In [ ]:
import numpy as np

vocab = ["battery", "great", "poor", "price", "screen"]   # a 5-word vocabulary
target_word = "great"

one_hot = np.zeros(len(vocab))
one_hot[vocab.index(target_word)] = 1.0
print(one_hot, "<- 'great'")

# Now scale up to realistic sizes:
V = 50_000          # typical English vocabulary after cleaning
words_per_doc = 150 # a long review

share_nonzero = words_per_doc / V
print(f"a {words_per_doc}-word document lights {share_nonzero:.2%} of a {V:,}d vector")
print(f"that vector stored densely: {V * 8 / 1024:.0f} KB per document - mostly zeros!")

## 2. Bag-of-Words (BoW)

Instead of one word per vector, BoW gives each **document** a vector whose slots count word occurrences. Grammar and order vanish; only "which words appear, how often" survives. scikit-learn's `CountVectorizer` learns the vocabulary (`fit`) and converts any text (`transform`) — and returns a sparse matrix so those zeros cost nothing.

**Syntax:**

```python
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()              # options: lowercase, stop_words, ngram_range...
X = vectorizer.fit_transform(list_of_texts) # sparse document-term matrix
vectorizer.vocabulary_                      # dict: word -> column index
vectorizer.get_feature_names_out()          # column index -> word
X.toarray()                                 # dense view (small matrices only!)
```

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

DOCS = [
    "great battery great screen",
    "poor battery poor speaker",
    "great speaker great price",
    "poor screen poor price",
    "great battery poor speaker",
    "screen great battery price",
]

bow = CountVectorizer()
X = bow.fit_transform(DOCS)

print("shape:", X.shape, "-> 6 documents x", X.shape[1], "vocabulary words")
pairs = sorted(bow.vocabulary_.items(), key=lambda kv: kv[1])
print("vocabulary_:", pairs)
print("feature names:", list(bow.get_feature_names_out()))
print(X.toarray())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

DOCS = [
    "great battery great screen",
    "poor battery poor speaker",
    "great speaker great price",
    "poor screen poor price",
    "great battery poor speaker",
    "screen great battery price",
]
X = CountVectorizer().fit_transform(DOCS).toarray()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(X, cmap="Blues", vmin=0)
ax.set_xticks(range(X.shape[1]), CountVectorizer().fit(DOCS).get_feature_names_out(),
              rotation=90)
ax.set_yticks(range(len(DOCS)), [f"doc{i}" for i in range(len(DOCS))])
for i in range(X.shape[0]):                 # annotate every cell with its count
    for j in range(X.shape[1]):
        ax.text(j, i, X[i, j], ha="center", va="center",
                color="white" if X[i, j] > X.max() / 2 else "#333333")
fig.colorbar(im, ax=ax, label="count")
ax.set_title("Document-term matrix (Bag-of-Words)")
plt.tight_layout()
plt.show()

> 🔍 **Under the Hood:** `fit_transform` does NOT return a NumPy array — it returns a scipy **CSR sparse matrix**: three parallel arrays holding only nonzero values plus column indices. That is why `.toarray()` exists (and why calling it on a big corpus explodes RAM). Do the maths once and you never forget:

In [ ]:
n_docs, vocab_size, nnz_per_doc = 500_000, 50_000, 80

dense_bytes = n_docs * vocab_size * 8                     # float64 everywhere
sparse_bytes = n_docs * nnz_per_doc * (8 + 4)             # value + int32 column index

print(f"dense storage : {dense_bytes / 1024**3:8.1f} GB")
print(f"sparse storage: {sparse_bytes / 1024**3:8.2f} GB")
print(f"compression factor: ~{dense_bytes / sparse_bytes:,.0f}x")

## 3. BoW's Blind Spot: Word Order

Because bags ignore sequence, sentences with opposite meanings collapse into the same vector. This is the price of simplicity — and the reason n-grams exist (below).

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

headline_a = "dog bites man"      # ordinary news
headline_b = "man bites dog"      # extraordinary news!

X = CountVectorizer().fit_transform([headline_a, headline_b]).toarray()
print(X)
print("identical vectors?", np.array_equal(X[0], X[1]), "- BoW cannot tell them apart")

## 4. Fixing It Partly: N-grams

An **n-gram** bundles n consecutive words into one token. With `ngram_range=(1,2)` the bigrams `"dog bites"` and `"man bites"` differ, restoring a slice of word order — at the price of a much larger vocabulary.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

pair = ["dog bites man", "man bites dog"]

uni = CountVectorizer(ngram_range=(1, 1)).fit(pair)
bi = CountVectorizer(ngram_range=(1, 2)).fit(pair)

print("unigrams:", list(uni.get_feature_names_out()))
print("unigram vectors identical?",
      np.array_equal(uni.transform(pair).toarray()[0], uni.transform(pair).toarray()[1]))

print("\nwith bigrams:", list(bi.get_feature_names_out()))
print("bigram vectors identical?",
      np.array_equal(bi.transform(pair).toarray()[0], bi.transform(pair).toarray()[1]))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

DOCS = [
    "great battery great screen",
    "poor battery poor speaker",
    "great speaker great price",
    "poor screen poor price",
    "great battery poor speaker",
    "screen great battery price",
]

for rng in [(1, 1), (1, 2), (1, 3)]:
    vec = CountVectorizer(ngram_range=rng).fit(DOCS)
    sample = list(vec.get_feature_names_out())[-4:]
    print(f"ngram_range={rng}: {vec.get_feature_names_out().size:>3} features | e.g. {sample}")

### 4.1 Character N-grams: Robust Against Typos

Switching the `analyzer` from `"word"` to `"char"` makes overlapping letter chunks the tokens. Misspellings share most of their chunks with the correct spelling — so `"batery"` stays numerically close to `"battery"`, something word-level BoW completely misses.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

words = ["battery", "batery", "bananas"]

char_vec = CountVectorizer(analyzer="char", ngram_range=(2, 4))
M = char_vec.fit_transform(words)
sim = cosine_similarity(M)

names = ["battery", "batery ", "bananas"]
for i in range(len(words)):
    for j in range(i + 1, len(words)):
        print(f"sim({names[i].strip()}, {names[j]}): {sim[i, j]:.2f}")
# the typo stays close to 'battery'; 'bananas' drifts far away

## 5. TF-IDF, Decomposed By Hand

Raw counts flatter words that appear in *every* document (`is`, `the`). **TF-IDF** fixes this with two intuitions:

- **Term Frequency** `tf(t, d)` — how often word `t` appears in document `d` (locally important?)
- **Inverse Document Frequency** `idf(t) = ln((1+N)/(1+df)) + 1` — rare across the corpus ⇒ informative (globally distinctive?). Here `N` = number of documents, `df` = number of documents containing `t`. (This is exactly scikit-learn's `smooth_idf=True` default.)

Multiply them: `tfidf(t, d) = tf(t, d) * idf(t)`. Frequent-in-this-doc AND rare-everywhere wins.

Our toy corpus — four tiny documents:

In [ ]:
import numpy as np
import pandas as pd

TOY = [
    "battery life is great",
    "battery life is poor",
    "screen is great",
    "price is fair",
]
terms = sorted({t for d in TOY for t in d.split()})

tf = np.array([[d.count(t) for t in terms] for d in TOY], dtype=float)
pd.DataFrame(tf, index=[f"doc{i}" for i in range(4)], columns=terms).astype(int)

In [ ]:
import numpy as np
import pandas as pd

TOY = [
    "battery life is great",
    "battery life is poor",
    "screen is great",
    "price is fair",
]
terms = sorted({t for d in TOY for t in d.split()})
N = len(TOY)

df_counts = np.array([sum(t in d.split() for d in TOY) for t in terms])
idf = np.log((1 + N) / (1 + df_counts)) + 1     # sklearn smooth_idf formula

pd.DataFrame({"in_n_docs (df)": df_counts, "idf": np.round(idf, 3)}, index=terms)
# 'is' appears everywhere -> idf 1.0 ; 'fair' only once -> idf 1.92

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

TOY = [
    "battery life is great",
    "battery life is poor",
    "screen is great",
    "price is fair",
]
terms = sorted({t for d in TOY for t in d.split()})
N = len(TOY)
df_counts = np.array([sum(t in d.split() for d in TOY) for t in terms])
idf = np.log((1 + N) / (1 + df_counts)) + 1
tf = np.array([[d.count(t) for t in terms] for d in TOY], dtype=float)

hand_tfidf = tf * idf                            # our by-hand matrix

sk_tfidf = TfidfVectorizer(norm=None)            # norm=None -> skip final scaling
sk_matrix = sk_tfidf.fit_transform(TOY).toarray()

print("agreement with TfidfVectorizer:", np.allclose(hand_tfidf, sk_matrix))
pd.DataFrame(np.round(hand_tfidf, 3),
             index=[f"doc{i}" for i in range(N)], columns=terms)

By default `TfidfVectorizer` also applies **L2 normalization** — each document vector is scaled to length 1, so a 30-word review compares fairly against a 5-word tweet. Check it:

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

TOY = [
    "battery life is great",
    "battery life is poor",
    "screen is great",
    "price is fair",
]
default_matrix = TfidfVectorizer().fit_transform(TOY)   # norm='l2' by default
lengths = np.linalg.norm(default_matrix.toarray(), axis=1)
print("row lengths:", np.round(lengths, 6), "-> every document now has length 1")

## 6. Cosine Similarity — and a Mini Search Engine

Two documents match if they point the *same direction*, regardless of length. Cosine of the angle between vectors `u` and `v`: `cos(u,v) = (u . v) / (||u|| ||v||)` — 1 means identical direction, 0 unrelated. Because TF-IDF rows come pre-normalized, this reduces to a plain dot product.

**Syntax:**

```python
def cosine(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

scores = [cosine(query_vec, doc_vec) for doc_vec in doc_vectors]
ranked = sorted(scores, reverse=True)       # best match first
```

In [ ]:
import numpy as np

def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

u = np.array([2.0, 0.0, 1.0])    # "about phones"
v = np.array([4.0, 0.0, 2.0])    # same direction, twice as long
w = np.array([0.0, 3.0, 0.0])    # different topic

print("cos(u, v) =", round(cosine(u, v), 3), "(identical direction)")
print("cos(u, w) =", round(cosine(u, w), 3), "(orthogonal = unrelated)")

Now assemble the pieces into a search engine over six authored gadget blurbs: vectorize documents once, treat each user query as a mini-document, rank by cosine similarity.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

SEARCH_DOCS = {
    "phone_x ": "All-day battery life and fast charging make this phone a travel favourite.",
    "camera_z": "The camera shoots sharp photos at night without any flash.",
    "slim_pro": "Bright screen with slim bezels, though battery endurance is just average.",
    "boom_box": "Loud speakers and surprisingly rich bass for such a tiny size.",
    "gamer_one": "Battery drains quickly during gaming and charging feels slow.",
    "daily_pad": "Crisp display and smooth performance for everyday apps.",
}

engine = TfidfVectorizer()
doc_matrix = engine.fit_transform(list(SEARCH_DOCS.values())).toarray()

def search(query, k=3):
    q = engine.transform([query]).toarray()[0]
    scored = [(name, cosine(q, doc)) for name, doc in zip(SEARCH_DOCS, doc_matrix)]
    return sorted(scored, key=lambda pair: pair[1], reverse=True)[:k]

for query in ["long battery life", "sharp night photos", "good screen"]:
    print(f"\nquery: '{query}'")
    for name, score in search(query):
        print(f"   {score:.3f}  {name.strip()}")

scikit-learn ships optimized versions of both operations. When vectors are already L2-normalized (TF-IDF default), a plain dot-product kernel equals cosine similarity exactly — handy confirmation that our hand-rolled math was right.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

SEARCH_DOCS = {
    "phone_x ": "All-day battery life and fast charging make this phone a travel favourite.",
    "camera_z": "The camera shoots sharp photos at night without any flash.",
    "slim_pro": "Bright screen with slim bezels, though battery endurance is just average.",
    "boom_box": "Loud speakers and surprisingly rich bass for such a tiny size.",
    "gamer_one": "Battery drains quickly during gaming and charging feels slow.",
    "daily_pad": "Crisp display and smooth performance for everyday apps.",
}

engine = TfidfVectorizer()
docs = engine.fit_transform(list(SEARCH_DOCS.values()))
query = engine.transform(["long battery life"])

ours = linear_kernel(query, docs)                       # dot products
theirs = cosine_similarity(query, docs)                 # full cosine
print("max difference:", np.abs(ours - theirs).max())
print("identical result:", np.allclose(ours, theirs), "- normalized rows make them equal")

## 7. Mini Project — SMS Spam Classifier

Time to ship something. We author 24 messages (10 spam, 14 ham), wrap vectorizer and model in ONE `Pipeline` (so preprocessing can never diverge between training and prediction), and evaluate properly: accuracy **plus** precision/recall plus a confusion matrix — because for spam, false positives (real messages blocked) hurt differently than false negatives.

**Example:**

In [ ]:
MESSAGES = [
    # --- spam ---
    "WINNER! You have been selected for a $1000 gift card. Claim now!",
    "URGENT: your mobile number won a $5000 prize. Call 09099 now",
    "Free entry to win a brand new car! Text WIN to 80085",
    "Congratulations! You are our lucky customer today. Click link to claim",
    "Your account expires tomorrow. Update your details immediately here",
    "Cheap loans approved in minutes. No credit check! Reply YES",
    "Exclusive deal! Buy 1 get 5 free, today only. Shop now!!!",
    "Your parcel could not be delivered. Pay redelivery fee here: bit.ly/payfee",
    "Earn $500 weekly from home. No experience needed. Register today",
    "Claim your free holiday voucher before midnight. Limited offer!",
    # --- ham ---
    "Hey, running 10 minutes late, order the starters without me",
    "Can you send me the photos from yesterday? They look great",
    "Meeting moved to 3pm tomorrow, conference room B",
    "Thanks for the birthday wishes, had a wonderful day",
    "Did you take your umbrella? It is pouring here",
    "The movie started slow but the ending was brilliant",
    "Mum arrives at the airport at 6:40, flight BA217",
    "Left my charger at your place, bring it tonight?",
    "Doctor appointment confirmed for Friday at 10am",
    "That restaurant was amazing, booking again next weekend",
    "Finished the assignment finally. Never again haha",
    "Call me when you land, need directions to the hotel",
    "Match cancelled, pitch is waterlogged. Practice Thursday instead",
    "Your prescription is ready for pickup until 6pm",
]
LABELS = ["spam"] * 10 + ["ham"] * 14
print(len(MESSAGES), "messages |", LABELS.count("spam"), "spam,", LABELS.count("ham"), "ham")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

MESSAGES_SPAM = [
    "WINNER! You have been selected for a $1000 gift card. Claim now!",
    "URGENT: your mobile number won a $5000 prize. Call 09099 now",
    "Free entry to win a brand new car! Text WIN to 80085",
    "Congratulations! You are our lucky customer today. Click link to claim",
    "Your account expires tomorrow. Update your details immediately here",
    "Cheap loans approved in minutes. No credit check! Reply YES",
    "Exclusive deal! Buy 1 get 5 free, today only. Shop now!!!",
    "Your parcel could not be delivered. Pay redelivery fee here: bit.ly/payfee",
    "Earn $500 weekly from home. No experience needed. Register today",
    "Claim your free holiday voucher before midnight. Limited offer!",
    "Hey, running 10 minutes late, order the starters without me",
    "Can you send me the photos from yesterday? They look great",
    "Meeting moved to 3pm tomorrow, conference room B",
    "Thanks for the birthday wishes, had a wonderful day",
    "Did you take your umbrella? It is pouring here",
    "The movie started slow but the ending was brilliant",
    "Mum arrives at the airport at 6:40, flight BA217",
    "Left my charger at your place, bring it tonight?",
    "Doctor appointment confirmed for Friday at 10am",
    "That restaurant was amazing, booking again next weekend",
    "Finished the assignment finally. Never again haha",
    "Call me when you land, need directions to the hotel",
    "Match cancelled, pitch is waterlogged. Practice Thursday instead",
    "Your prescription is ready for pickup until 6pm",
]
Y_LABELS = ["spam"] * 10 + ["ham"] * 14

X_train, X_test, y_train, y_test = train_test_split(
    MESSAGES_SPAM, Y_LABELS, test_size=0.25, stratify=Y_LABELS, random_state=42,
)

spam_clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("naive_bayes", MultinomialNB(alpha=0.5)),
])
spam_clf.fit(X_train, y_train)

predictions = spam_clf.predict(X_test)
print(f"accuracy: {accuracy_score(y_test, predictions):.2f} "
      f"on {len(y_test)} held-out messages\n")
print(classification_report(y_test, predictions, target_names=["ham", "spam"], digits=2))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

MESSAGES_SPAM = [
    "WINNER! You have been selected for a $1000 gift card. Claim now!",
    "URGENT: your mobile number won a $5000 prize. Call 09099 now",
    "Free entry to win a brand new car! Text WIN to 80085",
    "Congratulations! You are our lucky customer today. Click link to claim",
    "Your account expires tomorrow. Update your details immediately here",
    "Cheap loans approved in minutes. No credit check! Reply YES",
    "Exclusive deal! Buy 1 get 5 free, today only. Shop now!!!",
    "Your parcel could not be delivered. Pay redelivery fee here: bit.ly/payfee",
    "Earn $500 weekly from home. No experience needed. Register today",
    "Claim your free holiday voucher before midnight. Limited offer!",
    "Hey, running 10 minutes late, order the starters without me",
    "Can you send me the photos from yesterday? They look great",
    "Meeting moved to 3pm tomorrow, conference room B",
    "Thanks for the birthday wishes, had a wonderful day",
    "Did you take your umbrella? It is pouring here",
    "The movie started slow but the ending was brilliant",
    "Mum arrives at the airport at 6:40, flight BA217",
    "Left my charger at your place, bring it tonight?",
    "Doctor appointment confirmed for Friday at 10am",
    "That restaurant was amazing, booking again next weekend",
    "Finished the assignment finally. Never again haha",
    "Call me when you land, need directions to the hotel",
    "Match cancelled, pitch is waterlogged. Practice Thursday instead",
    "Your prescription is ready for pickup until 6pm",
]
Y_LABELS = ["spam"] * 10 + ["ham"] * 14

X_train, X_test, y_train, y_test = train_test_split(
    MESSAGES_SPAM, Y_LABELS, test_size=0.25, stratify=Y_LABELS, random_state=42,
)
clf = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                ("naive_bayes", MultinomialNB(alpha=0.5))])
clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

cm = confusion_matrix(y_test, predictions, labels=["ham", "spam"])
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["ham", "spam"])
ax.set_yticks([0, 1], ["ham", "spam"])
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("Spam classifier confusion matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "#333333")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

Finally, inspect individual verdicts. On tiny datasets there may be no mistakes at all — so a good habit is to look at whichever predictions were **least confident**, those are the ones to worry about first.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

MESSAGES_SPAM = [
    "WINNER! You have been selected for a $1000 gift card. Claim now!",
    "URGENT: your mobile number won a $5000 prize. Call 09099 now",
    "Free entry to win a brand new car! Text WIN to 80085",
    "Congratulations! You are our lucky customer today. Click link to claim",
    "Your account expires tomorrow. Update your details immediately here",
    "Cheap loans approved in minutes. No credit check! Reply YES",
    "Exclusive deal! Buy 1 get 5 free, today only. Shop now!!!",
    "Your parcel could not be delivered. Pay redelivery fee here: bit.ly/payfee",
    "Earn $500 weekly from home. No experience needed. Register today",
    "Claim your free holiday voucher before midnight. Limited offer!",
    "Hey, running 10 minutes late, order the starters without me",
    "Can you send me the photos from yesterday? They look great",
    "Meeting moved to 3pm tomorrow, conference room B",
    "Thanks for the birthday wishes, had a wonderful day",
    "Did you take your umbrella? It is pouring here",
    "The movie started slow but the ending was brilliant",
    "Mum arrives at the airport at 6:40, flight BA217",
    "Left my charger at your place, bring it tonight?",
    "Doctor appointment confirmed for Friday at 10am",
    "That restaurant was amazing, booking again next weekend",
    "Finished the assignment finally. Never again haha",
    "Call me when you land, need directions to the hotel",
    "Match cancelled, pitch is waterlogged. Practice Thursday instead",
    "Your prescription is ready for pickup until 6pm",
]
Y_LABELS = ["spam"] * 10 + ["ham"] * 14

X_train, X_test, y_train, y_test = train_test_split(
    MESSAGES_SPAM, Y_LABELS, test_size=0.25, stratify=Y_LABELS, random_state=42,
)
clf = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
                ("naive_bayes", MultinomialNB(alpha=0.5))])
clf.fit(X_train, y_train)

proba_spam = clf.predict_proba(X_test)[:, 1]        # classes_ sorts alphabetically
results = pd.DataFrame({
    "message": X_test,
    "true": y_test,
    "predicted": clf.predict(X_test),
    "p(spam)": np.round(proba_spam, 3),
})
errors = results[results["true"] != results["predicted"]]

if len(errors):
    print("misclassified:")
    print(errors.to_string(index=False))
else:
    print("no misclassifications - least confident verdicts instead:")
    print(results.reindex(results["p(spam)"].abs().sub(0.5).abs().sort_values().index)
                 .head(3).to_string(index=False))

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Calling `fit_transform` on the test set too | test vocabulary leaks into training stats — inflated scores | fit on train only; `Pipeline` enforces it |
| Comparing raw BoW counts across documents | longer docs look "more similar" to everything | use TF-IDF (L2-normalized) instead |
| Reading `.vocabulary_` backwards | it maps word → **column index**, not the reverse | use `.get_feature_names_out()` |
| Trusting accuracy on imbalanced spam data | 86% accuracy by predicting "ham" always | read precision/recall per class + confusion matrix |
| Expecting cosine similarity above 1 | cosine measures angle, capped at 1 | >1 signals unnormalized vectors or a bug |

## ⚠️ Gotcha: `toarray()` Is a Trap

`X.toarray()` on a real corpus allocates `docs × vocab` floats — the 500 GB lesson above. Inspect small slices (`X[:20].toarray()`) and keep pipelines operating on the sparse object.

## 💡 Best Practices & Pro Tips

- Prefer `TfidfVectorizer` over raw counts whenever documents vary in length.
- Start with `ngram_range=(1, 2)` and `min_count`-style filtering (`min_df=2`) before growing the vocabulary.
- Wrap vectorizer + estimator in a `Pipeline`; it is the cheapest insurance against train/predict skew.
- **AI-engineering relevance:** everything here scales straight into production search: TF-IDF + cosine is still the first stage of real retrieval systems (and of RAG pipelines), where cheap sparse ranking narrows millions of documents down to hundreds before a neural reranker looks at them.

## 📌 Summary

| Tool / Concept | What it does | Example |
|---|---|---|
| `CountVectorizer` | words → count vectors | `.fit_transform(docs)` |
| `.vocabulary_` | word → column index dict | `{'battery': 0, ...}` |
| `TfidfVectorizer` | counts weighted by rarity | `tf * log((1+N)/(1+df)) + 1` |
| `norm='l2'` (default) | equal-length document vectors | fair short-vs-long comparison |
| `analyzer='char'` | letter-chunk tokens | tolerates `batery` |
| `ngram_range=(1, 2)` | add word pairs | keeps some word order |
| `cosine(u, v)` | angle between vectors | 1 = same direction |
| `Pipeline([...])` | chained fit/transform | vectorizer + Naive Bayes |
| `MultinomialNB` | classic text classifier | fast, works on counts/TF-IDF |

**Key takeaways**

- Representation choice matters as much as the model: BoW → TF-IDF → n-grams trade simplicity for signal.
- TF-IDF is two ideas multiplied: local frequency × global rarity — you can compute it on paper.
- Cosine similarity turns vectors into a search engine in ten lines.
- Evaluate with precision/recall/confusion, never accuracy alone, on imbalanced problems.

## 🔗 Next Lesson

These vectors know what words *appear* — not what they *mean*. Next we give words geometry with meaning baked in: [04_Word_Embeddings](../04_Word_Embeddings/notes.ipynb).